# 00. Setup — 환경 초기화
**프로젝트**: 화물차 DTG 위험운전행동의 법적 심각도 자동 분류를 위한 교통법령 KG 기반 IRAC-V GraphRAG 프레임워크

| 순서 | 내용 |
|------|------|
| 1 | 환경변수 확인 (.env) |
| 2 | 의존성 설치 |
| 3 | 디렉토리 구조 확인 |
| 4 | 법제처 API 연결 테스트 |

In [5]:
import sys, subprocess, pathlib, os, platform
from dotenv import load_dotenv
load_dotenv()

REQUIRED = {
    'LAW_OC': '법제처 API 인증키',
    'GITHUB_TOKEN': 'GitHub PAT',
    'NEO4J_PW': 'Neo4j PW',
    'ANTHROPIC_API_KEY': 'Anthropic API',
    'OPENAI_API_KEY': 'OpenAI Embedding',
}

print('=== 환경변수 ===')
for var, desc in REQUIRED.items():
    val = os.getenv(var, '')
    print(f'  {"✅" if val else "❌"} {var:22s}: {val[:4]+"****" if val else "미설정 ("+desc+")"}')

print(f'\n=== 런타임 ===')
print(f'  Python: {sys.version.split()[0]}  |  OS: {platform.system()}')
print(f'  루트: {pathlib.Path().resolve()}')
print(f'  .env: {"✅" if pathlib.Path(".env").exists() else "❌"}')

=== 환경변수 ===
  ✅ LAW_OC                : zhzh****
  ✅ GITHUB_TOKEN          : ghp_****
  ✅ NEO4J_PW              : 2sll****
  ✅ ANTHROPIC_API_KEY     : sk-a****
  ✅ OPENAI_API_KEY        : sk-p****

=== 런타임 ===
  Python: 3.12.7  |  OS: Windows
  루트: C:\Users\pc\OneDrive\Desktop\프로젝트\dtg-legal-graphrag
  .env: ✅


## 2. 의존성 설치

In [6]:
PKGS = ['python-dotenv','pandas','numpy','requests','neo4j',
        'anthropic','matplotlib','scikit-learn','tqdm',
        'faiss-cpu','openai','scipy']
print('=== 패키지 설치 ===')
for p in PKGS:
    r = subprocess.run([sys.executable,'-m','pip','install',p,'-q'], capture_output=True)
    print(f'  {"✅" if r.returncode==0 else "❌"} {p}')

=== 패키지 설치 ===
  ✅ python-dotenv
  ✅ pandas
  ✅ numpy
  ✅ requests
  ✅ neo4j
  ✅ anthropic
  ✅ matplotlib
  ✅ scikit-learn
  ✅ tqdm
  ✅ faiss-cpu
  ✅ openai
  ✅ scipy


## 3. 디렉토리 구조

In [7]:
from config import ALL_DIRS, DTG_DIR, SPEED_DIR, LAW_DIR, STAT_DIR, REF_DIR

print('=== 데이터 폴더 ===')
for n, p in [('DTG',DTG_DIR),('제한속도',SPEED_DIR),('법령',LAW_DIR),
             ('사고통계',STAT_DIR),('참고',REF_DIR)]:
    cnt = len(list(p.glob('*'))) if p.exists() else 0
    print(f'  {"✅" if cnt else "⚠️"} {n:8s}: {p} ({cnt}개)')

print('\n=== 출력 폴더 ===')
for d in ALL_DIRS:
    e = d.exists()
    d.mkdir(parents=True, exist_ok=True)
    print(f'  {"(기존)" if e else "(신규)"} {d}')

=== 데이터 폴더 ===
  ✅ DTG     : data\01_DTG_RAW (5개)
  ✅ 제한속도    : data\02_SPEED_LIMIT (2개)
  ✅ 법령      : data\03_LAW_DATA (15개)
  ✅ 사고통계    : data\04_ACCIDENT_STAT (2개)
  ✅ 참고      : data\05_REFERENCE (2개)

=== 출력 폴더 ===
  (기존) data\01_DTG_RAW
  (기존) data\02_SPEED_LIMIT
  (기존) data\03_LAW_DATA
  (기존) data\04_ACCIDENT_STAT
  (기존) data\05_REFERENCE
  (기존) data\03_LAW_DATA\json
  (기존) data\03_LAW_DATA\penalty_tables
  (기존) data\scenarios
  (기존) data\kg
  (기존) results\stage1
  (기존) results\stage2
  (기존) results\stage3
  (기존) results\stage4
  (신규) results\evaluation
  (기존) figures
  (기존) utils


## 4. 법제처 API 연결 테스트

In [8]:
import requests, json

OC = os.getenv('LAW_OC', '')
if not OC:
    print('❌ LAW_OC 환경변수가 없습니다')
else:
    r = requests.get('https://www.law.go.kr/DRF/lawSearch.do',
                     params={'OC':OC,'target':'law','type':'JSON','query':'도로교통법','display':3},
                     timeout=15)
    data = json.loads(r.text)
    laws = data.get('LawSearch',{}).get('law',[])
    if not isinstance(laws, list): laws = [laws]
    print(f'✅ API 정상 | {len(laws)}건 검색됨')
    for law in laws[:3]:
        print(f'   {law.get("법령명한글","?")} (MST={law.get("법령일련번호","?")})')
    print('\n→ 다음: 01_data_collection.py')

✅ API 정상 | 3건 검색됨
   도로교통법 (MST=281875)
   도로교통법 시행령 (MST=269989)
   도로교통법 시행규칙 (MST=285317)

→ 다음: 01_data_collection.py
